## 1) Make sure all of your installs and imports are good.

In [1]:
import pandas as pd
import os
import glob
import re
from pathlib import Path
import numpy as np
from bokeh.plotting import figure, show
from bokeh.layouts import gridplot
from bokeh.models import HoverTool
from bokeh.io import output_notebook

## 2) Reading in all of the files
Make sure that your input files are properly in their folders. This code is for tecplot files, but could be modified for hd5

In [2]:
def extract_tec_files(working_directory):
    work_dir = Path(working_directory)

    if not work_dir.exists():
        print(f"Directory {working_directory} does not exist!")
        return {}

    # Find all .tec files with the pattern name-XXX.tec
    tec_files = glob.glob(str(work_dir / "*.tec"))

    # Dictionary to store grouped files
    file_groups = {}

    # Pattern to match files like somename-000.tec, somename-001.tec
    pattern = re.compile(r"^(.+)-(\d{3})\.tec$")

    for file_path in tec_files:
        filename = os.path.basename(file_path)
        match = pattern.match(filename)

        if match:
            base_name = match.group(1)
            sequence_num = int(match.group(2))

            if base_name not in file_groups:
                file_groups[base_name] = []

            file_groups[base_name].append((sequence_num, file_path))

    # Sort files by number for each base name
    for base_name in file_groups:
        file_groups[base_name].sort(key=lambda x: x[0])
        # Keep file paths
        file_groups[base_name] = [file_path for _, file_path in file_groups[base_name]]

    return file_groups

In [4]:
# Example usage:
working_directory = "/Users/sanskriti/Documents/GitHub/saltyBiomass/software_module/pflotran_vis15/modified_pflotran_files_test1"

tec_file_groups = extract_tec_files(working_directory)

# Display results
for base_name, files in tec_file_groups.items():
    print(f"\nBase name: {base_name}")
    print(f"Number of files: {len(files)}")
    for i, file_path in enumerate(files):
        print(f"  {i:03d}: {os.path.basename(file_path)}")


Base name: 9_addnitrogen_11M
Number of files: 51
  000: 9_addnitrogen_11M-000.tec
  001: 9_addnitrogen_11M-001.tec
  002: 9_addnitrogen_11M-002.tec
  003: 9_addnitrogen_11M-003.tec
  004: 9_addnitrogen_11M-004.tec
  005: 9_addnitrogen_11M-005.tec
  006: 9_addnitrogen_11M-006.tec
  007: 9_addnitrogen_11M-007.tec
  008: 9_addnitrogen_11M-008.tec
  009: 9_addnitrogen_11M-009.tec
  010: 9_addnitrogen_11M-010.tec
  011: 9_addnitrogen_11M-011.tec
  012: 9_addnitrogen_11M-012.tec
  013: 9_addnitrogen_11M-013.tec
  014: 9_addnitrogen_11M-014.tec
  015: 9_addnitrogen_11M-015.tec
  016: 9_addnitrogen_11M-016.tec
  017: 9_addnitrogen_11M-017.tec
  018: 9_addnitrogen_11M-018.tec
  019: 9_addnitrogen_11M-019.tec
  020: 9_addnitrogen_11M-020.tec
  021: 9_addnitrogen_11M-021.tec
  022: 9_addnitrogen_11M-022.tec
  023: 9_addnitrogen_11M-023.tec
  024: 9_addnitrogen_11M-024.tec
  025: 9_addnitrogen_11M-025.tec
  026: 9_addnitrogen_11M-026.tec
  027: 9_addnitrogen_11M-027.tec
  028: 9_addnitrogen_11M-0

## 3) We're going to extract water activity.

In [5]:
def extract_water_activity(tec_file_groups):
    water_activity_data = {}

    for base_name, files in tec_file_groups.items():

        first_file = files[0]
        # Read the tecplot file header to find variable names, find the variables line
        with open(first_file, "r") as f:
            lines = f.readlines()
        variables_line = None
        data_start_line = 0

        for i, line in enumerate(lines):
            if line.strip().startswith("VARIABLES"):
                variables_line = line
                break
            elif line.strip().startswith("ZONE"):
                data_start_line = i + 1
                break
        if variables_line:
            var_matches = re.findall(r'"([^"]*)"', variables_line)
            column_names = var_matches
            # Skip to data section (after ZONE line)
            for i, line in enumerate(lines):
                if line.strip().startswith("ZONE"):
                    data_start_line = i + 1
                    break
            # Read data starting from data_start_line
            df = pd.read_csv(
                first_file,
                sep="\s+",
                skiprows=data_start_line,
                header=None,
                names=column_names,
            )

        # Check if gamma_H2O column
        if "Gamma H2O" in df.columns:
            water_activity_data[base_name] = {
                "file": first_file,
                "Gamma H2O": df["Gamma H2O"].values,
                "coordinates": {
                    "x": df["X"].values if "X" in df.columns else None,
                    "y": df["Y"].values if "Y" in df.columns else None,
                    "z": df["Z"].values if "Z" in df.columns else None,
                },
            }
            print(f"✓ Extracted water activity from {base_name}: {len(df)} data points")
        else:
            print("???? something went wrong. check if you have Gamma H2O")
    return water_activity_data


# Extract water activity from all 000 files
water_activity_results = extract_water_activity(tec_file_groups)

# Display summary
print(f"Successfully extracted from {len(water_activity_results)} file groups:")
for base_name, data in water_activity_results.items():
    gamma_values = data["Gamma H2O"]
    print(
        f"  {base_name}: min={gamma_values.min():.4f}, max={gamma_values.max():.4f}, mean={gamma_values.mean():.4f}"
    )

✓ Extracted water activity from 9_addnitrogen_11M: 64 data points
✓ Extracted water activity from 9_addnitrogen_10M: 64 data points
✓ Extracted water activity from 9_addnitrogen_5M: 64 data points
✓ Extracted water activity from 9_addnitrogen_4M: 64 data points
✓ Extracted water activity from 9_addnitrogen_1M: 64 data points
✓ Extracted water activity from 9_addnitrogen_20M: 64 data points
✓ Extracted water activity from 9_addnitrogen_17M: 64 data points
✓ Extracted water activity from 9_addnitrogen_16M: 64 data points
✓ Extracted water activity from 9_addnitrogen_14M: 64 data points
✓ Extracted water activity from 9_addnitrogen_15M: 64 data points
✓ Extracted water activity from 9_addnitrogen_2M: 64 data points
✓ Extracted water activity from 9_addnitrogen_3M: 64 data points
✓ Extracted water activity from 9_addnitrogen_9M: 64 data points
✓ Extracted water activity from 9_addnitrogen_8M: 64 data points
✓ Extracted water activity from 9_addnitrogen_7M: 64 data points
✓ Extracted water 

<>:29: SyntaxWarning: invalid escape sequence '\s'
<>:29: SyntaxWarning: invalid escape sequence '\s'
/var/folders/9z/tkcd7rfx1bxd05989nnkhc7r0000gn/T/ipykernel_40234/4152188528.py:29: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(first_file, sep='\s+', skiprows=data_start_line,


## 4a) (dummy) let me try just getting the carbon and concentrations at the top

In [6]:
def extract_surface_concentrations(tec_file_groups):
    surface_concentration_data = {}

    for base_name, files in tec_file_groups.items():
        print(f"\nProcessing {base_name}...")
        base_data = {"files": [], "time_series": []}

        for file_idx, file_path in enumerate(files):
            try:
                # Read the tecplot file header to find variable names
                with open(file_path, "r") as f:
                    lines = f.readlines()

                # Parse tecplot header
                variables_line = None
                data_start_line = 0

                for i, line in enumerate(lines):
                    if line.strip().startswith("VARIABLES"):
                        variables_line = line
                    elif line.strip().startswith("ZONE"):
                        data_start_line = i + 1
                        break

                if not variables_line:
                    print(
                        f"  ✗ Could not find VARIABLES line in {os.path.basename(file_path)}"
                    )
                    continue

                # Extract column names and read data
                column_names = re.findall(r'"([^"]*)"', variables_line)
                df = pd.read_csv(
                    file_path,
                    sep=r"\s+",
                    skiprows=data_start_line,
                    header=None,
                    names=column_names,
                )

                # Look for the specific columns based on the file format
                ch4_col = None
                co2_col = None

                # Look for CH4 column (try both Free and Total)
                if "Free CH4(aq) [M]" in df.columns:
                    ch4_col = "Free CH4(aq) [M]"
                elif "Total CH4(aq) [M]" in df.columns:
                    ch4_col = "Total CH4(aq) [M]"
                else:
                    print(f"  ✗ No CH4 column found in {os.path.basename(file_path)}")
                    print(
                        f"     Available columns: {list(df.columns[:10])}..."
                    )  # Show first 10 columns
                    continue

                # Look for CO2 column (try both Free and Total)
                if "Free CO2(aq) [M]" in df.columns:
                    co2_col = "Free CO2(aq) [M]"
                elif "Total CO2(aq) [M]" in df.columns:
                    co2_col = "Total CO2(aq) [M]"
                else:
                    print(f"  ✗ No CO2 column found in {os.path.basename(file_path)}")
                    continue

                if "Z [m]" not in df.columns:
                    print(f"  ✗ No Z coordinate found in {os.path.basename(file_path)}")
                    print(
                        f"     Available columns: {list(df.columns[:10])}..."
                    )  # Show first 10 columns
                    continue

                # Find the highest Z coordinate plane (surface)
                max_z = df["Z [m]"].max()
                surface_mask = df["Z [m]"] == max_z
                surface_data = df[surface_mask]

                # Extract concentrations
                file_data = {
                    "file_index": file_idx,
                    "file_name": os.path.basename(file_path),
                    "max_z": max_z,
                    "n_surface_points": len(surface_data),
                    "coordinates": {
                        "x": (
                            surface_data["X [m]"].values
                            if "X [m]" in df.columns
                            else None
                        ),
                        "y": (
                            surface_data["Y [m]"].values
                            if "Y [m]" in df.columns
                            else None
                        ),
                        "z": surface_data["Z [m]"].values,
                    },
                }

                file_data["CH4_concentration"] = surface_data[ch4_col].values
                file_data["CH4_column"] = ch4_col

                file_data["CO2_concentration"] = surface_data[co2_col].values
                file_data["CO2_column"] = co2_col

                base_data["time_series"].append(file_data)
                print(
                    f"  ✓ Successfully processed {os.path.basename(file_path)} (CH4: {ch4_col}, CO2: {co2_col})"
                )

            except Exception as e:
                print(f"  ✗ Error reading {os.path.basename(file_path)}: {e}")
                import traceback

                traceback.print_exc()  # This will show the full error traceback

        if base_data["time_series"]:
            surface_concentration_data[base_name] = base_data
            print(
                f"  ✓ Successfully processed {len(base_data['time_series'])} files for {base_name}"
            )

    return surface_concentration_data


# Run the function again
surface_concentrations = extract_surface_concentrations(tec_file_groups)


Processing 9_addnitrogen_11M...
  ✓ Successfully processed 9_addnitrogen_11M-000.tec (CH4: Free CH4(aq) [M], CO2: Free CO2(aq) [M])
  ✓ Successfully processed 9_addnitrogen_11M-001.tec (CH4: Free CH4(aq) [M], CO2: Free CO2(aq) [M])
  ✓ Successfully processed 9_addnitrogen_11M-002.tec (CH4: Free CH4(aq) [M], CO2: Free CO2(aq) [M])
  ✓ Successfully processed 9_addnitrogen_11M-003.tec (CH4: Free CH4(aq) [M], CO2: Free CO2(aq) [M])
  ✓ Successfully processed 9_addnitrogen_11M-004.tec (CH4: Free CH4(aq) [M], CO2: Free CO2(aq) [M])
  ✓ Successfully processed 9_addnitrogen_11M-005.tec (CH4: Free CH4(aq) [M], CO2: Free CO2(aq) [M])
  ✓ Successfully processed 9_addnitrogen_11M-006.tec (CH4: Free CH4(aq) [M], CO2: Free CO2(aq) [M])
  ✓ Successfully processed 9_addnitrogen_11M-007.tec (CH4: Free CH4(aq) [M], CO2: Free CO2(aq) [M])
  ✓ Successfully processed 9_addnitrogen_11M-008.tec (CH4: Free CH4(aq) [M], CO2: Free CO2(aq) [M])
  ✓ Successfully processed 9_addnitrogen_11M-009.tec (CH4: Free CH4

## 4) We're going to get carbon and methane flux from the files for Day 1, 2, and 3, and then plot it

In [7]:
output_notebook()


def calculate_flux_and_plot(water_activity_results, surface_concentrations, max_days=4):
    """
    Calculate flux and plot water activity vs CO2/CH4 flux for the first few days

    Parameters:
    water_activity_results: Dictionary from extract_water_activity function
    surface_concentrations: Dictionary from extract_surface_concentrations function
    max_days: Number of days to plot (default 4)
    """

    # Prepare data for plotting
    plot_data = {}

    for base_name in water_activity_results.keys():
        if base_name in surface_concentrations:
            # Get water activity (mean from 000 file)
            water_activity = water_activity_results[base_name]["Gamma H2O"].mean()

            # Get time series data
            time_series = surface_concentrations[base_name]["time_series"]

            # Calculate fluxes for first few days (files)
            days = []
            ch4_fluxes = []
            co2_fluxes = []

            for i, time_data in enumerate(time_series[:max_days]):
                if (
                    "CH4_concentration" in time_data
                    and "CO2_concentration" in time_data
                ):
                    # Assuming 1 m^2 surface area and converting concentration to flux
                    # Simple approximation: flux = mean_concentration * surface_area
                    surface_area = 1.0  # m^2

                    ch4_mean = time_data["CH4_concentration"].mean()
                    co2_mean = time_data["CO2_concentration"].mean()

                    # Convert from M to mol/m^2 (assuming unit depth for simplification)
                    ch4_flux = ch4_mean * surface_area  # mol/m^2
                    co2_flux = co2_mean * surface_area  # mol/m^2

                    days.append(i)
                    ch4_fluxes.append(ch4_flux)
                    co2_fluxes.append(co2_flux)

            plot_data[base_name] = {
                "water_activity": water_activity,
                "days": days,
                "ch4_fluxes": ch4_fluxes,
                "co2_fluxes": co2_fluxes,
            }

    # Create plots for each day
    plots = []
    colors = ["red", "blue", "green", "orange", "purple", "brown", "pink", "gray"]

    for day in range(max_days):
        # CH4 plot
        p1 = figure(
            width=400,
            height=300,
            title=f"Day {day}: Water Activity vs CH4 Flux",
            x_axis_label="Water Activity (Gamma H2O)",
            y_axis_label="CH4 Flux (mol/m²)",
        )

        # CO2 plot
        p2 = figure(
            width=400,
            height=300,
            title=f"Day {day}: Water Activity vs CO2 Flux",
            x_axis_label="Water Activity (Gamma H2O)",
            y_axis_label="CO2 Flux (mol/m²)",
        )

        # Collect data for this day
        water_activities = []
        ch4_fluxes_day = []
        co2_fluxes_day = []
        base_names = []

        for i, (base_name, data) in enumerate(plot_data.items()):
            if day < len(data["days"]):
                water_activities.append(data["water_activity"])
                ch4_fluxes_day.append(data["ch4_fluxes"][day])
                co2_fluxes_day.append(data["co2_fluxes"][day])
                base_names.append(base_name)

        if water_activities:  # Only plot if we have data
            color = colors[day % len(colors)]

            # Add hover tool
            hover1 = HoverTool(
                tooltips=[
                    ("Base Name", "@base_name"),
                    ("Water Activity", "@x{0.0000}"),
                    ("CH4 Flux", "@y{0.00e0}"),
                ]
            )
            hover2 = HoverTool(
                tooltips=[
                    ("Base Name", "@base_name"),
                    ("Water Activity", "@x{0.0000}"),
                    ("CO2 Flux", "@y{0.00e0}"),
                ]
            )

            p1.add_tools(hover1)
            p2.add_tools(hover2)

            # Plot CH4
            source1 = dict(x=water_activities, y=ch4_fluxes_day, base_name=base_names)
            p1.circle("x", "y", size=8, color=color, alpha=0.7, source=source1)

            # Plot CO2
            source2 = dict(x=water_activities, y=co2_fluxes_day, base_name=base_names)
            p2.circle("x", "y", size=8, color=color, alpha=0.7, source=source2)

        plots.append([p1, p2])

    grid = gridplot(plots)
    show(grid)


calculate_flux_and_plot(water_activity_results, surface_concentrations, max_days=4)

Loading BokehJS ...

## 5) Also plotting TOTAL escaping

In [8]:
def calculate_cumulative_emissions_and_plot(
    water_activity_results, surface_concentrations
):

    # Define time periods of interest (in days)
    target_days = [1, 3, 10, 30, 50, 100, 300, 1000, 3000, 10000, 30000]

    # Prepare data for plotting
    plot_data = {}

    for base_name in water_activity_results.keys():
        if base_name in surface_concentrations:
            # Get water activity (mean from 000 file)
            water_activity = water_activity_results[base_name]["Gamma H2O"].mean()

            # Get time series data
            time_series = surface_concentrations[base_name]["time_series"]

            # Calculate cumulative emissions
            cumulative_ch4 = []
            cumulative_co2 = []
            available_days = []

            total_ch4 = 0
            total_co2 = 0

            for i, time_data in enumerate(time_series):
                if (
                    "CH4_concentration" in time_data
                    and "CO2_concentration" in time_data
                ):
                    # Calculate daily emissions (assuming 1 m^2 surface area)
                    surface_area = 1.0  # m^2

                    ch4_mean = time_data["CH4_concentration"].mean()
                    co2_mean = time_data["CO2_concentration"].mean()

                    # Daily flux (mol/m^2/day)
                    daily_ch4 = ch4_mean * surface_area
                    daily_co2 = co2_mean * surface_area

                    # Add to cumulative totals
                    total_ch4 += daily_ch4
                    total_co2 += daily_co2

                    cumulative_ch4.append(total_ch4)
                    cumulative_co2.append(total_co2)
                    available_days.append(i)

            # Find cumulative emissions at target time periods
            target_emissions = {}

            for target_day in target_days:
                if target_day < len(available_days):
                    target_emissions[target_day] = {
                        "ch4": cumulative_ch4[target_day],
                        "co2": cumulative_co2[target_day],
                    }

            plot_data[base_name] = {
                "water_activity": water_activity,
                "target_emissions": target_emissions,
                "cumulative_ch4": cumulative_ch4,
                "cumulative_co2": cumulative_co2,
                "available_days": available_days,
            }

    # Create plots for each target time period
    plots = []
    colors = [
        "red",
        "blue",
        "green",
        "orange",
        "purple",
        "brown",
        "pink",
        "gray",
        "cyan",
        "magenta",
    ]

    # Determine which time periods have data
    available_periods = set()
    for data in plot_data.values():
        available_periods.update(data["target_emissions"].keys())

    available_periods = sorted(list(available_periods))

    print(f"Available time periods: {available_periods} days")

    # Create plots in rows of 2
    for i in range(0, len(available_periods), 2):
        row_plots = []

        for j in range(2):
            if i + j < len(available_periods):
                target_day = available_periods[i + j]
                color = colors[j % len(colors)]

                # CH4 plot
                p1 = figure(
                    width=400,
                    height=300,
                    title=f"Day {target_day}: Water Activity vs Cumulative CH4",
                    x_axis_label="Water Activity (Gamma H2O)",
                    y_axis_label="Cumulative CH4 (mol/m²)",
                )

                # CO2 plot
                p2 = figure(
                    width=400,
                    height=300,
                    title=f"Day {target_day}: Water Activity vs Cumulative CO2",
                    x_axis_label="Water Activity (Gamma H2O)",
                    y_axis_label="Cumulative CO2 (mol/m²)",
                )

                # Collect data for this time period
                water_activities = []
                ch4_emissions = []
                co2_emissions = []
                base_names = []

                for base_name, data in plot_data.items():
                    if target_day in data["target_emissions"]:
                        water_activities.append(data["water_activity"])
                        ch4_emissions.append(
                            data["target_emissions"][target_day]["ch4"]
                        )
                        co2_emissions.append(
                            data["target_emissions"][target_day]["co2"]
                        )
                        base_names.append(base_name)

                if water_activities:  # Only plot if we have data
                    # Add hover tools
                    hover1 = HoverTool(
                        tooltips=[
                            ("Base Name", "@base_name"),
                            ("Water Activity", "@x{0.0000}"),
                            ("Cumulative CH4", "@y{0.00e0}"),
                        ]
                    )
                    hover2 = HoverTool(
                        tooltips=[
                            ("Base Name", "@base_name"),
                            ("Water Activity", "@x{0.0000}"),
                            ("Cumulative CO2", "@y{0.00e0}"),
                        ]
                    )

                    p1.add_tools(hover1)
                    p2.add_tools(hover2)

                    # Plot data
                    source1 = dict(
                        x=water_activities, y=ch4_emissions, base_name=base_names
                    )
                    source2 = dict(
                        x=water_activities, y=co2_emissions, base_name=base_names
                    )

                    p1.circle("x", "y", size=8, color=color, alpha=0.7, source=source1)
                    p2.circle("x", "y", size=8, color=color, alpha=0.7, source=source2)

                row_plots.extend([p1, p2])

        if row_plots:
            plots.append(row_plots)

    # Show plots
    grid = gridplot(plots)
    show(grid)

    # Print summary statistics
    print("\n=== Cumulative Emissions Summary ===")
    for base_name, data in plot_data.items():
        print(f"\n{base_name} (Water Activity: {data['water_activity']:.4f}):")
        for target_day in sorted(data["target_emissions"].keys()):
            emissions = data["target_emissions"][target_day]
            print(
                f"  Day {target_day:5d}: CH4={emissions['ch4']:.2e} mol/m², CO2={emissions['co2']:.2e} mol/m²"
            )


# Calculate and plot cumulative emissions
calculate_cumulative_emissions_and_plot(water_activity_results, surface_concentrations)

Available time periods: [1, 3, 10, 30, 50] days



=== Cumulative Emissions Summary ===

9_addnitrogen_11M (Water Activity: 0.7153):
  Day     1: CH4=2.25e-15 mol/m², CO2=2.19e-04 mol/m²
  Day     3: CH4=4.25e-15 mol/m², CO2=2.57e-04 mol/m²
  Day    10: CH4=1.13e-14 mol/m², CO2=3.91e-04 mol/m²
  Day    30: CH4=3.13e-14 mol/m², CO2=7.72e-04 mol/m²
  Day    50: CH4=5.13e-14 mol/m², CO2=1.15e-03 mol/m²

9_addnitrogen_10M (Water Activity: 0.7332):
  Day     1: CH4=2.26e-15 mol/m², CO2=2.19e-04 mol/m²
  Day     3: CH4=4.26e-15 mol/m², CO2=2.57e-04 mol/m²
  Day    10: CH4=1.13e-14 mol/m², CO2=3.91e-04 mol/m²
  Day    30: CH4=3.13e-14 mol/m², CO2=7.72e-04 mol/m²
  Day    50: CH4=5.13e-14 mol/m², CO2=1.15e-03 mol/m²

9_addnitrogen_5M (Water Activity: 0.8233):
  Day     1: CH4=2.55e-15 mol/m², CO2=2.19e-04 mol/m²
  Day     3: CH4=4.55e-15 mol/m², CO2=2.57e-04 mol/m²
  Day    10: CH4=1.16e-14 mol/m², CO2=3.91e-04 mol/m²
  Day    30: CH4=3.16e-14 mol/m², CO2=7.72e-04 mol/m²
  Day    50: CH4=5.16e-14 mol/m², CO2=1.15e-03 mol/m²

9_addnitrogen_4M 